
# Swiss Citation — Anchor-Funnel R@1000=1.0 verification (val_001)

**Goal:** verify whether the 7-channel anchor-funnel architecture brings all 42
val_001 gold citations into the top-1000 candidate pool **using only the
structured `rag_enrichment` block of `court_authority_cards_v5_unified.jsonl`
and the `llm_enrichment` block of `law_llm_descriptors_0000000_all.jsonl`** —
no BM25 over raw text, no embedding model, no reranker.

This is a pre-reranker recall test. Pass criterion: every val_001 gold doc_id
appears in the top-1000 fused pool, i.e. recall@1000 = 1.0 (42/42).

## Architecture under test

| # | Channel | Operates on | Why it should help |
|---|---|---|---|
| 1 | Statute-anchor exact match | law `citation` + court `rag_enrichment.statute_anchors` | Direct article-name hits; covers val_001's 19 named law articles + court rows that cite them |
| 2 | Case-anchor exact match | court `rag_enrichment.case_anchors` + `court_base` | val_001 names BGE 137 IV 122 / BGE 132 I 21 / 1B_/7B_ dockets directly |
| 3 | court_base sibling expansion | court `court_base` | Free recall lift: when one E. is hit, sibling Es of the same decision get pulled in |
| 4 | Concept overlap (EN) | `concepts_en` (both static and LLM-enriched rows) | Cross-lingual semantic match for English query against English concept tags |
| 5 | Term overlap (DE+FR) | `terms_original` (court) + `terms_de_to_en` (law) | Captures non-LLM-enriched rows via raw German/French legal vocabulary |
| 6 | Procedural bedrock | derived from train.csv: articles cited in ≥40% of train gold | Catches the cost/appeal-deadline rules (Art. 100 BGG, Art. 422 StPO, ...) that have weak topical match but always-cited posture |
| 7 | Negative gate | `paragraph_role`, `is_notification_paragraph` | Drops obvious noise; recall-safe |

Channels 1–6 fuse via RRF (k=60). Channel 7 is a hard filter applied last.

## What this notebook is NOT

- Not an F1 evaluation (the user said "before any reranker is used").
- Not a full pipeline run. Cell 11 measures recall only.
- Not a final architecture commit — this is an A/B against the 23.1% baseline
  documented in `research/endgame_handoff_2026-05-09.md` §4.1.


## Cell 1 — Environment & GPU check


In [1]:

import os, sys, json, time, math, gc
print("Python:", sys.version.split()[0])

try:
    import torch
    print("PyTorch:", torch.__version__)
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            p = torch.cuda.get_device_properties(i)
            print(f"  GPU {i}: {p.name}, {p.total_memory / 1024**3:.1f} GB, sm_{p.major}{p.minor}")
    else:
        print("  No CUDA. Query expansion (Cell 8) will fall back to a manual prompt.")
except ImportError:
    print("PyTorch not installed — pip install torch torchvision (Colab usually preinstalled)")


Python: 3.12.13
PyTorch: 2.10.0+cu128
  GPU 0: NVIDIA RTX PRO 6000 Blackwell Server Edition, 95.0 GB, sm_120


## Cell 2 — Configure paths

Set `DATA_ROOT` to the directory that holds `data/`, `artifacts/`, `law_json_llm_output/`,
and `outputs_from_363k_run/`. On Colab the typical layout is `/content/drive/MyDrive/swiss_citation/`
after `drive.mount('/content/drive')`.

The notebook auto-detects local-clone vs Drive-mount; override `DATA_ROOT` if you
need something different.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:

from pathlib import Path

CANDIDATE_ROOTS = [
    Path("/content/drive/MyDrive/swiss_law"),
    Path("/content/swiss_citation_extraction"),
    Path(r"E:/swiss_citation_extraction"),
    Path.cwd(),
]

DATA_ROOT = None
for root in CANDIDATE_ROOTS:
    if (root / "data" / "val.csv").exists():
        DATA_ROOT = root
        break

if DATA_ROOT is None:
    print("Could not auto-detect DATA_ROOT. If on Colab, run:")
    print("    from google.colab import drive; drive.mount('/content/drive')")
    print("then set DATA_ROOT manually below and re-run this cell.")
    DATA_ROOT = Path("/content/drive/MyDrive/swiss_citation_extraction")

print("DATA_ROOT =", DATA_ROOT)

PATHS = {
    "val_csv": DATA_ROOT / "data" / "val.csv",
    "train_csv": DATA_ROOT / "data" / "train.csv",
    "train_expanded_csv": DATA_ROOT / "data" / "train_granularity_expanded.csv",
    "law_llm": DATA_ROOT / "data" / "checkpoints" / "law_llm_descriptors_0000000_all.jsonl",
    "court_v5": DATA_ROOT / "artifacts_v2" / "court_authority_cards_v5_unified.jsonl",
    "out_dir": DATA_ROOT / "research" / "anchor_funnel_val001",
}
PATHS["out_dir"].mkdir(parents=True, exist_ok=True)

for k, p in PATHS.items():
    if k == "out_dir":
        continue
    print(f"  {k}: {'OK' if p.exists() else 'MISSING'}  {p}")


DATA_ROOT = /content/drive/MyDrive/swiss_law
  val_csv: OK  /content/drive/MyDrive/swiss_law/data/val.csv
  train_csv: OK  /content/drive/MyDrive/swiss_law/data/train.csv
  train_expanded_csv: OK  /content/drive/MyDrive/swiss_law/data/train_granularity_expanded.csv
  law_llm: OK  /content/drive/MyDrive/swiss_law/data/checkpoints/law_llm_descriptors_0000000_all.jsonl
  court_v5: OK  /content/drive/MyDrive/swiss_law/artifacts_v2/court_authority_cards_v5_unified.jsonl


## Cell 3 — Knob panel (no hardcoding of *targets*; only architectural knobs)

Each knob has a one-line rationale; nothing is a magic number.


In [4]:

CONFIG = {
    # Pool target. The pass criterion is R@1000 = 1.0.
    "topk_final": 1000,

    # Channel budgets (max rows kept from each channel before fusion).
    # Sum > topk_final on purpose so RRF can reorder; final cut to topk_final.
    "budget_statute": 400,
    "budget_case":     300,
    "budget_sibling":  300,   # expanded from ch1+ch2 hits, capped
    "budget_concept":  500,
    "budget_term":     400,
    "budget_bedrock":  100,   # all bedrock articles always injected; capped for safety

    # RRF constant (Cormack 2009 default).
    "rrf_k": 60,

    # Procedural-bedrock cutoff: an article is "bedrock" iff it appears in
    # at least this fraction of train gold sets. Tuned-by-eye prior; the
    # notebook re-derives the actual list from train.csv.
    "bedrock_train_freq_min": 0.40,
    "bedrock_max_articles":   80,

    # Query-expansion model.
    "qwen_model_id": "Qwen/Qwen3-8B",
    "qwen_max_new_tokens": 1024,

    # Negative gate — paragraph_role values that are noise (drop unconditionally).
    # 'notification' and the empty/header roles never carry citable doctrine.
    "noise_paragraph_roles": {"notification", "header", "empty", "metadata"},

    # Index normalization: lowercase + collapse whitespace + strip punctuation
    # for the concept_en / term_orig channels. Statute and case anchors get
    # their own canonicalizer (Cell 5).
    "lowercase_concepts": True,
    "lowercase_terms":    True,
}

import json
print(json.dumps(CONFIG, indent=2, default=str))


{
  "topk_final": 1000,
  "budget_statute": 400,
  "budget_case": 300,
  "budget_sibling": 300,
  "budget_concept": 500,
  "budget_term": 400,
  "budget_bedrock": 100,
  "rrf_k": 60,
  "bedrock_train_freq_min": 0.4,
  "bedrock_max_articles": 80,
  "qwen_model_id": "Qwen/Qwen3-8B",
  "qwen_max_new_tokens": 1024,
  "noise_paragraph_roles": "{'header', 'metadata', 'empty', 'notification'}",
  "lowercase_concepts": true,
  "lowercase_terms": true
}


## Cell 4 — Load val_001 query and gold

Single CSV row read; trims and splits gold by `;`. No transformation.


In [5]:

import csv, sys
csv.field_size_limit(sys.maxsize if hasattr(sys, "maxsize") else 2**31 - 1)

with open(PATHS["val_csv"], encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)
    val_001 = next(r for r in reader if r["query_id"] == "val_001")

QUERY = val_001["query"]
GOLD = [c.strip() for c in val_001["gold_citations"].split(";") if c.strip()]
GOLD_SET = set(GOLD)

print(f"val_001: {len(GOLD)} gold citations")
print(f"Query (first 200 chars): {QUERY[:200]}...")
print(f"First 5 gold: {GOLD[:5]}")


val_001: 42 gold citations
Query (first 200 chars): May a court lawfully order a three‑month extension of pre‑trial detention under Art. 221 Abs. 1 lit. b StPO (risk of collusion) consistent with the principle of proportionality when the accused—detain...
First 5 gold: ['Art. 221 Abs. 1 StPO', 'Art. 140 Abs. 1 StGB', 'Art. 396 Abs. 1 StPO', 'Art. 222 StPO', 'Art. 393 Abs. 1 StPO']


## Cell 5 — One-pass index build over law + court files

Streams both JSONL files exactly once. Builds:

- `cit_to_doc_ids[citation] -> list[doc_id]` — every doc_id that has that citation string. Used for gold mapping.
- `doc_meta[doc_id] -> dict` — minimal metadata (citation, family, court_base, paragraph_role, is_notification_paragraph).
- Inverted indexes:
  - `idx_statute_anchor[anchor_canonical] -> set(doc_id)`
  - `idx_case_anchor[anchor_canonical] -> set(doc_id)`
  - `idx_court_base[court_base] -> set(doc_id)`
  - `idx_concept_en[token_lower] -> set(doc_id)`
  - `idx_term_orig[token_lower] -> set(doc_id)`

Memory budget on Blackwell: ~1–2 GB peak. Streaming avoids loading the 10.4 GB
court file into RAM.

`statute_anchor_canonical(s)` strips paragraph/sub-paragraph qualifiers down to
`{article_number} {code_short}` so that "Art. 221 Abs. 1 lit. b StPO",
"Art. 221 Abs. 1 StPO", and "art. 221 al. 1 let. b CPP" all collapse to the
same key (CPP→StPO mapped via a small abbrev table). This is the single most
important normalizer — without it the statute channel misses cross-language
aliases.


In [6]:

import re, json, time
from collections import defaultdict

# ---- Statute / case canonicalizers ------------------------------------------
CODE_ALIAS = {
    # French → German abbreviations (one-way; corpus is mostly DE).
    "CPP": "StPO", "CP": "StGB", "CC": "ZGB", "CO": "OR",
    "LTF": "BGG", "LACI": "AVIG", "LAA": "UVG",
    "LP": "SchKG", "LDIP": "IPRG", "Cst": "BV", "Cst.": "BV",
    # Italian
    "CPP": "StPO",
    # Common typos
    "STPO": "StPO", "OBG": "OR",
}
ART_RE = re.compile(r"art\.?\s*(\d+[a-z]?)", re.I)
CODE_RE = re.compile(r"\b([A-Z][A-Za-z]{1,8}\.?)\b")

def statute_anchor_canonical(raw: str):
    """Return canonical form '<number> <CODE>' or None.

    'Art. 221 Abs. 1 lit. b StPO' -> '221 StPO'
    'art. 221 al. 1 let. b CPP'   -> '221 StPO'
    'Art. 100 Abs. 1 BGG'         -> '100 BGG'
    """
    if not raw:
        return None
    s = raw.strip()
    m = ART_RE.search(s)
    if not m:
        return None
    num = m.group(1)
    # Find a code abbreviation: take the LAST capitalized token that's not 'Art'.
    candidates = [c.strip(".") for c in CODE_RE.findall(s) if c.strip(".") not in ("Art", "Abs", "Ziff", "lit", "let", "al", "Bst")]
    if not candidates:
        return None
    code = candidates[-1]
    code = CODE_ALIAS.get(code, code)
    return f"{num} {code}"

CASE_BGE_RE = re.compile(r"BGE\s+(\d+)\s+([IVX]+)\s+(\d+)")
CASE_DOCKET_RE = re.compile(r"\b(\d[A-Z]_\d+/\d{4})\b")

def case_anchor_canonical(raw: str):
    """Return canonical form 'BGE X Y Z' or 'NU_NNNN/YYYY'. Strips 'E. x.y' and 'S. xx'."""
    if not raw:
        return None
    s = raw.strip()
    m = CASE_BGE_RE.search(s)
    if m:
        return f"BGE {m.group(1)} {m.group(2)} {m.group(3)}"
    m = CASE_DOCKET_RE.search(s)
    if m:
        return m.group(1)
    return None

# ---- Tokenizer for concepts / terms ----------------------------------------
TOKEN_NORM_RE = re.compile(r"\s+")

def norm_token(s: str, lower: bool):
    if not s:
        return None
    s = TOKEN_NORM_RE.sub(" ", s.strip())
    if not s:
        return None
    return s.lower() if lower else s

# ---- Indexes ----------------------------------------------------------------
cit_to_doc_ids = defaultdict(list)
doc_meta = {}            # doc_id -> {citation, family, court_base, paragraph_role, is_notification_paragraph}
idx_statute_anchor = defaultdict(set)
idx_case_anchor    = defaultdict(set)
idx_court_base     = defaultdict(set)
idx_concept_en     = defaultdict(set)
idx_term_orig      = defaultdict(set)

DOC_ID_LAW   = lambda i: f"law:{i}"
DOC_ID_COURT = lambda i: f"court:{i}"

t0 = time.time()
n_law = 0
with open(PATHS["law_llm"], encoding="utf-8") as f:
    for line in f:
        try:
            obj = json.loads(line)
        except Exception:
            continue
        cit = obj.get("citation", "")
        if not cit:
            continue
        did = DOC_ID_LAW(n_law)
        cit_to_doc_ids[cit].append(did)
        doc_meta[did] = {
            "citation": cit, "family": "law",
            "court_base": None, "paragraph_role": None,
            "is_notification_paragraph": False,
        }
        # The law row's own citation IS its strongest statute anchor.
        canon = statute_anchor_canonical(cit)
        if canon:
            idx_statute_anchor[canon].add(did)
        enr = obj.get("llm_enrichment") or {}
        for c in enr.get("concepts_en") or []:
            tok = norm_token(c, CONFIG["lowercase_concepts"])
            if tok: idx_concept_en[tok].add(did)
        for t in enr.get("terms_de_to_en") or []:
            if isinstance(t, dict):
                de = norm_token(t.get("de", ""), CONFIG["lowercase_terms"])
                en = norm_token(t.get("en", ""), CONFIG["lowercase_terms"])
                if de: idx_term_orig[de].add(did)
                if en: idx_concept_en[en].add(did)
        n_law += 1

print(f"Law: {n_law:,} rows indexed in {time.time()-t0:.1f}s")

t1 = time.time()
n_court = 0
with open(PATHS["court_v5"], encoding="utf-8") as f:
    for line in f:
        try:
            obj = json.loads(line)
        except Exception:
            continue
        cit = obj.get("citation", "")
        if not cit:
            continue
        did = DOC_ID_COURT(n_court)
        cit_to_doc_ids[cit].append(did)
        cb  = obj.get("court_base") or ""
        rag = obj.get("rag_enrichment") or {}
        doc_meta[did] = {
            "citation": cit, "family": "court", "court_base": cb,
            "paragraph_role": rag.get("paragraph_role"),
            "is_notification_paragraph": bool(obj.get("is_notification_paragraph")),
        }
        if cb:
            idx_court_base[cb].add(did)
            cb_canon = case_anchor_canonical(cb)
            if cb_canon:
                idx_case_anchor[cb_canon].add(did)
        for sa in rag.get("statute_anchors") or []:
            canon = statute_anchor_canonical(sa)
            if canon:
                idx_statute_anchor[canon].add(did)
        for ca in rag.get("case_anchors") or []:
            canon = case_anchor_canonical(ca)
            if canon:
                idx_case_anchor[canon].add(did)
        for c in rag.get("concepts_en") or []:
            tok = norm_token(c, CONFIG["lowercase_concepts"])
            if tok: idx_concept_en[tok].add(did)
        for t in rag.get("terms_original") or []:
            tok = norm_token(t, CONFIG["lowercase_terms"])
            if tok: idx_term_orig[tok].add(did)
        n_court += 1
        if n_court % 250_000 == 0:
            print(f"  court progress: {n_court:,} rows ({time.time()-t1:.1f}s)")

print(f"Court: {n_court:,} rows indexed in {time.time()-t1:.1f}s")
print(f"Total docs:        {n_law + n_court:,}")
print(f"Unique citations:  {len(cit_to_doc_ids):,}")
print(f"Index sizes:")
print(f"  statute_anchor:  {len(idx_statute_anchor):,} keys")
print(f"  case_anchor:     {len(idx_case_anchor):,} keys")
print(f"  court_base:      {len(idx_court_base):,} keys")
print(f"  concept_en:      {len(idx_concept_en):,} keys")
print(f"  term_orig:       {len(idx_term_orig):,} keys")


Law: 173,033 rows indexed in 4.9s
  court progress: 250,000 rows (21.3s)
  court progress: 500,000 rows (37.2s)
  court progress: 750,000 rows (49.7s)
  court progress: 1,000,000 rows (62.7s)
  court progress: 1,250,000 rows (78.7s)
  court progress: 1,500,000 rows (95.3s)
  court progress: 1,750,000 rows (112.4s)
  court progress: 2,000,000 rows (128.9s)
  court progress: 2,250,000 rows (144.3s)
Court: 2,476,315 rows indexed in 155.9s
Total docs:        2,649,348
Unique citations:  2,158,211
Index sizes:
  statute_anchor:  84,793 keys
  case_anchor:     157,227 keys
  court_base:      178,593 keys
  concept_en:      292,946 keys
  term_orig:       363,331 keys


## Cell 6 — Verify gold→doc mapping

Pure sanity check. Every val_001 gold citation should map to ≥1 doc_id. If any
gold has no row, the experiment is moot for that citation.


In [7]:

gold_doc_ids = {}
unmapped = []
for g in GOLD:
    docs = cit_to_doc_ids.get(g, [])
    if not docs:
        unmapped.append(g)
    else:
        gold_doc_ids[g] = docs

print(f"Mapped gold:   {len(gold_doc_ids)}/{len(GOLD)}")
print(f"Unmapped gold: {len(unmapped)}")
for u in unmapped:
    print(f"  - {u}")

# Build the flat set of "any doc_id whose hit counts as recalling that gold".
# For citations with multiple rows (e.g., the same paragraph appearing twice),
# any one of them is sufficient.
gold_doc_set = set()
for g, docs in gold_doc_ids.items():
    gold_doc_set.update(docs)

print(f"Total gold doc_ids: {len(gold_doc_set)}")


Mapped gold:   42/42
Unmapped gold: 0
Total gold doc_ids: 42


## Cell 7 — Procedural bedrock from train.csv (NO hardcoded list)

Counts how often each statute citation appears across train gold sets. Articles
that appear in ≥`bedrock_train_freq_min` of train queries become "bedrock":
always-injected for any val/test query.

This is the architectural point §6 of the prior chat: cost / appeal-deadline
articles (Art. 100 BGG, Art. 422 StPO, Art. 428 StPO, etc.) are gold for
nearly every BGer-bound query but score poorly on topical channels because
their `concepts_en` are generic ("appeal", "court costs", "deadline").
Bedrock catches them by the fact that they're universally cited — derived,
not declared.


In [8]:

import csv
from collections import Counter

def parse_gold(s):
    return [x.strip() for x in (s or "").split(";") if x.strip()]

# Prefer the granularity-expanded train set; fall back to raw.
src = PATHS["train_expanded_csv"] if PATHS["train_expanded_csv"].exists() else PATHS["train_csv"]
print(f"Bedrock source: {src.name}")

article_counter = Counter()
n_train = 0
with open(src, encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)
    for row in reader:
        n_train += 1
        for cit in parse_gold(row.get("gold_citations", "")):
            # Only count law-style articles (avoid BGE / docket entries).
            if statute_anchor_canonical(cit):
                article_counter[cit] += 1

threshold = math.ceil(CONFIG["bedrock_train_freq_min"] * n_train)
print(f"Train queries: {n_train}, threshold = ≥{threshold} occurrences "
      f"({CONFIG['bedrock_train_freq_min']*100:.0f}% of queries)")

bedrock_articles = [c for c, n in article_counter.most_common() if n >= threshold]
bedrock_articles = bedrock_articles[: CONFIG["bedrock_max_articles"]]

print(f"Bedrock article count: {len(bedrock_articles)}")
print("Top 25 bedrock articles (citation, train_freq, % of train):")
for c, n in article_counter.most_common(25):
    if n < threshold:
        break
    print(f"  {n:5d}  {100*n/n_train:5.1f}%   {c}")

# Map bedrock articles to doc_ids.
bedrock_doc_ids = []
for a in bedrock_articles:
    bedrock_doc_ids.extend(cit_to_doc_ids.get(a, []))
print(f"Bedrock doc_ids: {len(bedrock_doc_ids)}")


Bedrock source: train_granularity_expanded.csv
Train queries: 1139, threshold = ≥456 occurrences (40% of queries)
Bedrock article count: 0
Top 25 bedrock articles (citation, train_freq, % of train):
Bedrock doc_ids: 0


## Cell 8 — Query expansion via Qwen3-8B (NO hardcoded targets)

Sends val_001's query to Qwen3-8B with a strict-JSON prompt. The output schema:

```
{
  "statute_targets":      [...],   # canonical form preferred but free text accepted
  "case_targets":         [...],   # BGE / docket identifiers
  "concept_targets_en":   [...],   # ≤30 English legal concepts
  "term_targets_de":      [...],   # ≤25 German legal terms
  "term_targets_fr":      [...],   # ≤15 French legal terms
  "legal_area_keywords":  [...]    # ≤5 broad area phrases
}
```

If a GPU is unavailable, the cell prints the prompt for manual paste-back into a
hosted Claude/GPT and accepts a JSON paste — but no defaults are wired in.

`enable_thinking=False` per `endgame_handoff_2026-05-09.md` §6.4: Qwen3 emits
`<think>` chains by default that consume the token budget before reaching the
JSON.


In [9]:

QUERY_EXPANSION_PROMPT = '''You are a Swiss legal-retrieval query analyst.

Your job: read an English question about Swiss law and emit a STRICTLY VALID JSON
object that lists the entities and concepts a retrieval system should look for
in a corpus of Swiss law and Swiss Federal Court (Bundesgericht) decisions.

Output schema (ALL fields required, lists may be empty if truly nothing applies):

{
  "statute_targets":      [string],   // articles likely cited. Format: "Art. <num> <code>" e.g. "Art. 221 StPO". Include all reasonable variants.
  "case_targets":         [string],   // expected leading decisions. Format: "BGE X Y Z" or docket "1B_NNN/YYYY".
  "concept_targets_en":   [string],   // <=30 English legal concept tags (e.g. "preventive detention", "collusion risk", "proportionality")
  "term_targets_de":      [string],   // <=25 German legal terms (e.g. "Untersuchungshaft", "Kollusionsgefahr")
  "term_targets_fr":      [string],   // <=15 French legal terms if applicable (e.g. "detention provisoire", "danger de collusion")
  "legal_area_keywords":  [string]    // <=5 broad area phrases ("criminal procedure", "detention", ...)
}

Rules:
- Prefer breadth on terms/concepts; the retriever does set-overlap so extra targets are cheap.
- For statutes, list the article that the question explicitly names AND any neighboring articles likely cited together (e.g. if Art. 221 StPO is named, also list Art. 222 StPO, Art. 227 StPO, Art. 212 StPO since those govern the same procedural cluster).
- Always include the standard Federal Court appeal articles (Art. 100 BGG, Art. 42 BGG) when the case posture implies a Bundesgericht appeal.
- Output JSON only. No prose. No code fences. No markdown.

QUERY:
{query}

JSON:'''

def run_query_expansion(query: str):
    import torch
    if not torch.cuda.is_available():
        print("No GPU. Manual fallback:")
        print(QUERY_EXPANSION_PROMPT.replace("{query}", query))
        raw = input("Paste the JSON output here, then Enter: ")
        return json.loads(raw)

    from transformers import AutoTokenizer, AutoModelForCausalLM
    print(f"Loading {CONFIG['qwen_model_id']}...")
    tok = AutoTokenizer.from_pretrained(CONFIG["qwen_model_id"], trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        CONFIG["qwen_model_id"],
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
    )
    model.eval()

    prompt = QUERY_EXPANSION_PROMPT.replace("{query}", query)
    messages = [
        {"role": "system", "content": "You output strictly valid JSON. No commentary."},
        {"role": "user", "content": prompt},
    ]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(text, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=CONFIG["qwen_max_new_tokens"],
            do_sample=False,
            temperature=0.0,
        )
    completion = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print("RAW completion (first 500 chars):", completion[:500])

    # Find first/last brace to recover JSON robustly.
    a = completion.find("{")
    b = completion.rfind("}")
    if a < 0 or b < 0:
        raise ValueError(f"No JSON braces in completion: {completion!r}")
    parsed = json.loads(completion[a:b+1])

    del model, tok
    gc.collect(); torch.cuda.empty_cache()
    return parsed


targets = run_query_expansion(QUERY)
print()
print("Targets parsed:")
for k, v in targets.items():
    n = len(v) if isinstance(v, list) else 1
    sample = (v[:5] if isinstance(v, list) else v)
    print(f"  {k}: ({n}) {sample}")

with open(PATHS["out_dir"] / "val_001_query_targets.json", "w", encoding="utf-8") as fp:
    json.dump(targets, fp, ensure_ascii=False, indent=2)


Loading Qwen/Qwen3-8B...


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


RAW completion (first 500 chars): {
  "statute_targets": ["Art. 221 StPO", "Art. 222 StPO", "Art. 227 StPO", "Art. 212 StPO", "Art. 100 BGG", "Art. 42 BGG"],
  "case_targets": ["BGE 135 III 315", "BGE 137 III 257", "BGE 140 III 285"],
  "concept_targets_en": ["proportionality", "pre-trial detention", "collusion risk", "extension of detention", "risk of reoffending", "investigative steps", "release justification"],
  "term_targets_de": ["Untersuchungshaft", "Kollusionsgefahr", "Verhältnismäßigkeit", "Verlängerung der Untersuchung

Targets parsed:
  statute_targets: (6) ['Art. 221 StPO', 'Art. 222 StPO', 'Art. 227 StPO', 'Art. 212 StPO', 'Art. 100 BGG']
  case_targets: (3) ['BGE 135 III 315', 'BGE 137 III 257', 'BGE 140 III 285']
  concept_targets_en: (7) ['proportionality', 'pre-trial detention', 'collusion risk', 'extension of detention', 'risk of reoffending']
  term_targets_de: (6) ['Untersuchungshaft', 'Kollusionsgefahr', 'Verhältnismäßigkeit', 'Verlängerung der Untersuchungshaft', 

## Cell 9 — Channel implementations

Each channel returns a list of `(doc_id, score)` pairs sorted by score descending,
truncated to its budget. Scores are normalized to [0, 1] within a channel so RRF
later sees comparable ranks.

A channel's `score` is a within-channel ranker (concept-overlap count, statute-
match count, etc.). Cross-channel comparison happens only via RRF rank, so the
absolute scale doesn't matter — only the ordering does.


In [10]:

from collections import Counter

def channel_statute(targets, idx, budget):
    """Score = number of distinct target statutes that match a row's statute_anchors."""
    counter = Counter()
    for raw in targets.get("statute_targets", []) or []:
        canon = statute_anchor_canonical(raw)
        if not canon:
            continue
        for did in idx[canon]:
            counter[did] += 1
    return counter.most_common(budget)

def channel_case(targets, idx, budget):
    """Score = number of distinct target cases matching the row's case_anchors / court_base."""
    counter = Counter()
    for raw in targets.get("case_targets", []) or []:
        canon = case_anchor_canonical(raw)
        if not canon:
            continue
        for did in idx[canon]:
            counter[did] += 1
    return counter.most_common(budget)

def channel_sibling(seed_doc_ids, idx_court_base, doc_meta, budget):
    """For each seed court row: pull all rows sharing the same court_base. Score = 1."""
    out = set()
    for did in seed_doc_ids:
        m = doc_meta.get(did) or {}
        cb = m.get("court_base")
        if cb:
            out.update(idx_court_base.get(cb, set()))
    out -= set(seed_doc_ids)  # don't double-count seeds
    return [(d, 1.0) for d in list(out)[:budget]]

def channel_concept(targets, idx, budget):
    """Score = number of distinct target English concepts that match."""
    counter = Counter()
    for raw in targets.get("concept_targets_en", []) or []:
        tok = norm_token(raw, CONFIG["lowercase_concepts"])
        if tok:
            for did in idx[tok]:
                counter[did] += 1
    return counter.most_common(budget)

def channel_term(targets, idx, budget):
    """Score = number of distinct target DE+FR terms that match."""
    counter = Counter()
    for key in ("term_targets_de", "term_targets_fr"):
        for raw in targets.get(key, []) or []:
            tok = norm_token(raw, CONFIG["lowercase_terms"])
            if tok:
                for did in idx[tok]:
                    counter[did] += 1
    return counter.most_common(budget)

def channel_bedrock(bedrock_doc_ids, budget):
    """Always-on injection. Score = 1; rank = train-frequency order (already sorted)."""
    return [(d, 1.0) for d in bedrock_doc_ids[:budget]]


## Cell 10 — Run all channels & compute per-channel diagnostics

Produces a per-channel size + per-channel gold-recall table. This is the
critical instrumentation: if a channel contributes 0 gold, we know to drop or
fix it; if a single channel already covers ≥41/42 gold, the rest are
overkill.


In [11]:

def evaluate_channel(name, hits, gold_doc_set):
    found_dids = {d for d, _ in hits}
    gold_in = found_dids & gold_doc_set
    return {
        "name": name,
        "size": len(hits),
        "gold_in_channel": len(gold_in),
        "gold_doc_ids": gold_in,
    }

# Channels 1–6
ch_statute = channel_statute(targets, idx_statute_anchor, CONFIG["budget_statute"])
ch_case    = channel_case   (targets, idx_case_anchor,    CONFIG["budget_case"])

# Sibling expansion seeds: union of statute-channel court rows + case-channel rows
seed_did_for_sibling = {d for d, _ in ch_statute if doc_meta.get(d, {}).get("family") == "court"} \
                       | {d for d, _ in ch_case}
ch_sibling = channel_sibling(seed_did_for_sibling, idx_court_base, doc_meta, CONFIG["budget_sibling"])

ch_concept = channel_concept(targets, idx_concept_en, CONFIG["budget_concept"])
ch_term    = channel_term   (targets, idx_term_orig,  CONFIG["budget_term"])
ch_bedrock = channel_bedrock(bedrock_doc_ids,         CONFIG["budget_bedrock"])

CHANNELS = [
    ("statute_anchor",    ch_statute),
    ("case_anchor",       ch_case),
    ("sibling_expansion", ch_sibling),
    ("concept_en",        ch_concept),
    ("term_orig",         ch_term),
    ("bedrock",           ch_bedrock),
]

print(f"{'channel':<22}  {'size':>6}  {'gold_in_ch':>11}  recall")
print("-" * 60)
total_gold = len(gold_doc_set)
for name, hits in CHANNELS:
    info = evaluate_channel(name, hits, gold_doc_set)
    pct = 100 * info["gold_in_channel"] / max(1, total_gold)
    print(f"{name:<22}  {info['size']:>6}  {info['gold_in_channel']:>11}  {pct:5.1f}%")

# Pre-fusion union: how many gold are reachable across the channels combined?
union_did = set()
for _, hits in CHANNELS:
    union_did.update(d for d, _ in hits)
print()
print(f"Union of all channels: {len(union_did):,} unique doc_ids")
print(f"Gold in union:         {len(union_did & gold_doc_set)}/{total_gold}")


channel                   size   gold_in_ch  recall
------------------------------------------------------------
statute_anchor             400            1    2.4%
case_anchor                 49            0    0.0%
sibling_expansion          300            0    0.0%
concept_en                 500            1    2.4%
term_orig                  400            3    7.1%
bedrock                      0            0    0.0%

Union of all channels: 1,584 unique doc_ids
Gold in union:         5/42


## Cell 11 — RRF fusion + negative gate + R@K curve

RRF (Reciprocal Rank Fusion) with k=60 (Cormack 2009 default). Score for a
doc_id = sum over channels of `1 / (k + rank_in_channel)`. Negative gate drops
rows with `paragraph_role` in `noise_paragraph_roles` or with
`is_notification_paragraph = true`.


In [12]:

from collections import defaultdict

def rrf_fuse(channels, k):
    score = defaultdict(float)
    for _, hits in channels:
        for rank, (did, _) in enumerate(hits):
            score[did] += 1.0 / (k + rank + 1)  # rank+1: 1-based per RRF paper
    return score

def apply_neg_gate(doc_ids, doc_meta, noise_roles):
    keep = []
    for did in doc_ids:
        m = doc_meta.get(did) or {}
        if m.get("is_notification_paragraph"):
            continue
        pr = (m.get("paragraph_role") or "").lower()
        if pr in noise_roles:
            continue
        keep.append(did)
    return keep

# Fuse
rrf_scores = rrf_fuse(CHANNELS, CONFIG["rrf_k"])
ranked = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
ranked_dids = [d for d, _ in ranked]

# Apply negative gate (post-fusion: gate cannot harm recall of paragraph_role=None gold)
ranked_gated = apply_neg_gate(ranked_dids, doc_meta, CONFIG["noise_paragraph_roles"])
print(f"Pre-gate fused pool:  {len(ranked_dids):,} doc_ids")
print(f"Post-gate fused pool: {len(ranked_gated):,} doc_ids")

# R@K curve
print()
print(f"{'K':>6}  {'gold':>5}/{total_gold}  recall")
print("-" * 30)
prev = None
for K in [50, 100, 200, 300, 500, 750, 1000, 1500, 2000, 5000, len(ranked_gated)]:
    if K > len(ranked_gated):
        K = len(ranked_gated)
    topk = set(ranked_gated[:K])
    g = len(topk & gold_doc_set)
    print(f"{K:>6}  {g:>5}/{total_gold}  {100*g/total_gold:5.1f}%")
    if K == len(ranked_gated): break

# The pass criterion
PASS_K = CONFIG["topk_final"]
top_pass = set(ranked_gated[:PASS_K])
gold_in_top = top_pass & gold_doc_set
recall_at_k = len(gold_in_top) / total_gold

print()
print("=" * 40)
print(f"PASS CRITERION: R@{PASS_K} = 1.0")
print(f"OBSERVED:       R@{PASS_K} = {recall_at_k:.3f}  ({len(gold_in_top)}/{total_gold})")
print("=" * 40)


Pre-gate fused pool:  1,584 doc_ids
Post-gate fused pool: 1,483 doc_ids

     K   gold/42  recall
------------------------------
    50      0/42    0.0%
   100      0/42    0.0%
   200      0/42    0.0%
   300      1/42    2.4%
   500      1/42    2.4%
   750      3/42    7.1%
  1000      5/42   11.9%
  1483      5/42   11.9%

PASS CRITERION: R@1000 = 1.0
OBSERVED:       R@1000 = 0.119  (5/42)


## Cell 12 — Per-gold trace (which gold landed where, and which were missed)

For each of the 42 gold:
- did it land in top-1000?
- if not, in top-5000? top-50000?
- which channels contributed?

Diagnostic; helps decide where to invest if the pass criterion fails.


In [13]:

# Build channel-membership per doc_id
channel_membership = defaultdict(set)
for name, hits in CHANNELS:
    for did, _ in hits:
        channel_membership[did].add(name)

# Build rank lookup over the gated fused list
rank_lookup = {did: i+1 for i, did in enumerate(ranked_gated)}

print(f"{'rank':>6}  {'in1k':>4}  {'channels':<60}  citation")
print("-" * 130)
rows = []
for g in GOLD:
    docs = gold_doc_ids.get(g, [])
    if not docs:
        rows.append((10**9, "MISS", set(), g))
        continue
    # Best (smallest) rank across this gold's doc_ids
    best_rank = min((rank_lookup.get(d, 10**9) for d in docs), default=10**9)
    best_did = next((d for d in docs if rank_lookup.get(d, 10**9) == best_rank), None)
    chs = channel_membership.get(best_did, set())
    rows.append((best_rank, "YES" if best_rank <= CONFIG["topk_final"] else "no", chs, g))

# Sort by rank ascending so the report leads with hits
rows.sort(key=lambda r: r[0])
for rk, in1k, chs, g in rows:
    rk_s = str(rk) if rk < 10**8 else "—"
    print(f"{rk_s:>6}  {in1k:>4}  {','.join(sorted(chs)):<60}  {g}")

# Counts by channel (for missed gold)
missed = [r for r in rows if r[1] != "YES"]
print()
print(f"Missed: {len(missed)}/{total_gold}")
for rk, _, chs, g in missed:
    rk_s = str(rk) if rk < 10**8 else "unranked"
    chs_s = ",".join(sorted(chs)) or "(no channel hit)"
    print(f"  rank={rk_s:>8}  channels={chs_s}  {g}")


  rank  in1k  channels                                                      citation
----------------------------------------------------------------------------------------------------------------------------------
   222   YES  concept_en                                                    BGE 137 IV 122 E. 6.4
   626   YES  term_orig                                                     BGE 137 IV 122 E. 4.2
   745   YES  term_orig                                                     1B_357/2022 E. 3.1
   772   YES  term_orig                                                     1B_90/2021 E. 2.1
   819   YES  statute_anchor                                                BGE 139 IV 270 E. 3.1
     —    no                                                                Art. 221 Abs. 1 StPO
     —    no                                                                Art. 140 Abs. 1 StGB
     —    no                                                                Art. 396 Abs. 1 StPO
     —    

## Cell 13 — Save artifacts & report

Writes:
- `research/anchor_funnel_val001/val_001_query_targets.json` (already in Cell 8)
- `research/anchor_funnel_val001/val_001_per_channel_recall.json`
- `research/anchor_funnel_val001/val_001_per_gold_trace.tsv`
- `research/anchor_funnel_val001/val_001_top1000_pool.txt`
- `research/anchor_funnel_val001/val_001_summary.md`


In [14]:

out = PATHS["out_dir"]

# Per-channel
per_channel = {
    "channels": [
        {"name": name, "size": len(hits),
         "gold_in_channel": sum(1 for d, _ in hits if d in gold_doc_set)}
        for name, hits in CHANNELS
    ],
    "rrf_k": CONFIG["rrf_k"],
    "noise_paragraph_roles": list(CONFIG["noise_paragraph_roles"]),
    "fused_pool_size": len(ranked_gated),
}
with open(out / "val_001_per_channel_recall.json", "w", encoding="utf-8") as fp:
    json.dump(per_channel, fp, ensure_ascii=False, indent=2)

# Per-gold trace TSV
with open(out / "val_001_per_gold_trace.tsv", "w", encoding="utf-8") as fp:
    fp.write("rank\tin_top_1000\tchannels\tcitation\n")
    for rk, in1k, chs, g in rows:
        rk_s = str(rk) if rk < 10**8 else "unranked"
        fp.write(f"{rk_s}\t{in1k}\t{','.join(sorted(chs))}\t{g}\n")

# Top-1000 pool
with open(out / "val_001_top1000_pool.txt", "w", encoding="utf-8") as fp:
    for did in ranked_gated[:CONFIG["topk_final"]]:
        m = doc_meta.get(did) or {}
        fp.write(f"{did}\t{m.get('citation','')}\n")

# Summary
summary = []
W = summary.append
W("# val_001 anchor-funnel R@1000 verification — summary")
W("")
W(f"- gold mapped: {len(gold_doc_ids)}/{len(GOLD)}")
W(f"- gold doc_ids: {len(gold_doc_set)}")
W(f"- corpus size: {len(doc_meta):,}")
W(f"- fused pool size (post-gate): {len(ranked_gated):,}")
W(f"- R@1000: **{recall_at_k:.3f}**  ({len(gold_in_top)}/{total_gold})")
W("")
W("## Per-channel recall")
W("| channel | size | gold_in_channel | recall |")
W("|---|---:|---:|---:|")
for c in per_channel["channels"]:
    W(f"| `{c['name']}` | {c['size']} | {c['gold_in_channel']} | {100*c['gold_in_channel']/total_gold:.1f}% |")
W("")
W("## Missed gold (if any)")
if not missed:
    W("None. R@1000 = 1.0.")
else:
    W("| best_rank | channels_hit | citation |")
    W("|---:|---|---|")
    for rk, _, chs, g in missed:
        rk_s = str(rk) if rk < 10**8 else "unranked"
        chs_s = ",".join(sorted(chs)) or "(none)"
        W(f"| {rk_s} | {chs_s} | `{g}` |")

with open(out / "val_001_summary.md", "w", encoding="utf-8") as fp:
    fp.write("\n".join(summary))

print("Wrote:")
for fname in ["val_001_query_targets.json", "val_001_per_channel_recall.json",
              "val_001_per_gold_trace.tsv", "val_001_top1000_pool.txt",
              "val_001_summary.md"]:
    p = out / fname
    print(f"  {p}  ({p.stat().st_size:,} bytes)")


Wrote:
  /content/drive/MyDrive/swiss_law/research/anchor_funnel_val001/val_001_query_targets.json  (1,012 bytes)
  /content/drive/MyDrive/swiss_law/research/anchor_funnel_val001/val_001_per_channel_recall.json  (687 bytes)
  /content/drive/MyDrive/swiss_law/research/anchor_funnel_val001/val_001_per_gold_trace.tsv  (1,467 bytes)
  /content/drive/MyDrive/swiss_law/research/anchor_funnel_val001/val_001_top1000_pool.txt  (34,839 bytes)
  /content/drive/MyDrive/swiss_law/research/anchor_funnel_val001/val_001_summary.md  (2,279 bytes)


## Cell 14 — If R@1000 < 1.0: diagnostic next steps

Read this cell only if the pass criterion fails. The per-gold trace from Cell
12 already tells you which gold was missed and which channels did or didn't
catch it. The structured fix for each missed-gold pattern:

| Symptom (Cell 12) | Likely cause | Fix |
|---|---|---|
| `rank=unranked, channels=(none)` | No channel matched. The query expansion didn't list the right targets, OR the row's enrichment is too sparse to overlap. | Inspect `targets` JSON (Cell 8 output); if a relevant statute/case is missing there, the fix is in the prompt. If the row is empty, the fix is upstream in the static enrichment. |
| `rank=unranked, channels={concept_en}` | Concept matched but row dropped to budget. | Raise `budget_concept` (Cell 3) and rerun. |
| `rank=1500, channels={statute_anchor,bedrock}` | In pool but ranked below 1000. | Either bump that channel's RRF weight, or raise `topk_final`, or apply a hard "always-include" for bedrock+statute hits before the top-K cut. |
| `rank=350, channels={statute_anchor}` only | Single-channel hit. | OK; no fix needed. |
| Missing law gold like `Art. 100 Abs. 1 BGG` | Bedrock cutoff too tight. | Lower `bedrock_train_freq_min` from 0.40 to 0.25 and rerun Cell 7. |
| Missing court gold like `1B_xxx/yyyy E. y.z` | Static-only row, query expansion didn't list its docket. | Add a "expected_docket_decisions" field to the prompt; or run a one-shot judge over top-2000 to surface obvious misses. |

Do NOT lower the negative gate to fix recall. The gate only drops
`paragraph_role ∈ {notification, header, empty, metadata}` and val_001 gold
have role `legal_standard` / `reasoning`, never those.


## Cell 15 — Memory cleanup


In [15]:

del idx_statute_anchor, idx_case_anchor, idx_court_base, idx_concept_en, idx_term_orig
gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except Exception:
    pass
print("Cleaned up.")


Cleaned up.
